In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import MDS
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import pdist, squareform

In [23]:
df = pd.read_csv('Sep0_hellinger.csv', sep=',')

In [24]:
# Keep only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

In [25]:
# Define distances

#NB distance
def NB_distance(x, y):
    numerator = np.sum(np.abs(x - y) * (np.abs(x) + np.abs(y)))
    denominator = np.sum(x**2 + y**2)
    return (numerator / denominator)*(0.5) if denominator != 0 else 0

In [26]:
# Bray-Curtis distance
def bray_curtis_distance(x, y):
    num = np.sum(np.abs(x - y))
    denom = np.sum(x + y)
    return num / denom if denom != 0 else 0

In [27]:
# Kulczynski distance
def kulczynski_distance(x, y):
    min_sum = np.sum(np.minimum(x, y))
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    if sum_x == 0 or sum_y == 0:
        return 1  # Max dissimilarity
    return 1 - 0.5 * ((min_sum / sum_x) + (min_sum / sum_y))

In [28]:
# Chord distance
def chord_distance(x, y):
    x_norm = x / np.sqrt(np.sum(x**2)) if np.sum(x**2) != 0 else x
    y_norm = y / np.sqrt(np.sum(y**2)) if np.sum(y**2) != 0 else y
    return np.linalg.norm(x_norm - y_norm)

In [29]:
# Chi-square distance
def chi_square_distance(x, y):
    # Oransal bolluk (pik)
    x_prop = x / np.sum(x) if np.sum(x) != 0 else np.zeros_like(x)
    y_prop = y / np.sum(y) if np.sum(y) != 0 else np.zeros_like(y)
    mean_prop = (x_prop + y_prop) / 2

    with np.errstate(divide='ignore', invalid='ignore'):
        dist = np.where(mean_prop == 0, 0, ((x_prop - y_prop) ** 2) / mean_prop)

    return np.sqrt(np.sum(dist))

In [ ]:
#Dunn indeks
def dunn_index(X, labels):

    distances = pairwise_distances(X)  
    clusters = np.unique(labels)

    # intra-cluster diameters
    intra_dists = []
    for c in clusters:
        indices = np.where(labels == c)[0]
        if len(indices) > 1:
            intra_dists.append(np.max(distances[np.ix_(indices, indices)]))
        else:
            intra_dists.append(0)  # tek nokta varsa çap = 0
    max_intra = np.max(intra_dists)

    # Minimum distances between pairs of clusters
    inter_dists = []
    for i in range(len(clusters)):
        for j in range(i+1, len(clusters)):
            idx_i = np.where(labels == clusters[i])[0]
            idx_j = np.where(labels == clusters[j])[0]
            inter_dists.append(np.min(distances[np.ix_(idx_i, idx_j)]))
    min_inter = np.min(inter_dists)

    # Dunn index
    return min_inter / max_intra if max_intra > 0 else 0


In [ ]:
# Clustering metrics with Agglomerative Clustering using complete linkage
# The number of clusters was fixed at 3 because the artificial data was generated using 3latent clusters.
def clustering_metrics_complete(distance_func, name):
    dist_matrix = squareform(pdist(numeric_df.values, metric=distance_func))
    coords = MDS(n_components=2, dissimilarity='precomputed', random_state=42).fit_transform(dist_matrix)
    clustering = AgglomerativeClustering(n_clusters=3, metric='precomputed', linkage='complete')
    labels = clustering.fit_predict(dist_matrix)
    silhouette = silhouette_score(coords, labels)
    db = davies_bouldin_score(coords, labels)
    ch = calinski_harabasz_score(coords, labels)
    # Calculate Dunn index
    dn = dunn_index(coords, labels) # Use the dunn_index function defined earlier

    return {
        "Method": name,
        "Silhouette": silhouette,
        "Davies-Bouldin": db,
        "Calinski-Harabasz": ch,
        "Dunn": dn
    }

In [32]:
# Run the analysis
results = [
    clustering_metrics_complete(NB_distance, "NB"),
    clustering_metrics_complete(bray_curtis_distance, "Bray-Curtis"),
    clustering_metrics_complete(kulczynski_distance, "Kulczynski"),
    clustering_metrics_complete(chord_distance, "Chord"),
    clustering_metrics_complete(chi_square_distance, "Chi-square")
]

c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(


In [ ]:
#In Google Colab, some results after the decimal could be different
results_df = pd.DataFrame(results)

from IPython.display import display
display(results_df)

,Method,Silhouette,Davies-Bouldin,Calinski-Harabasz,Dunn
0,NB,0.550870,0.673271,250.071693,0.031343
1,Bray-Curtis,0.520236,0.682164,253.430940,0.049597
2,Kulczynski,0.507023,0.744309,208.326877,0.024573
3,Chord,0.487012,0.806271,187.746969,0.024050
4,Chi-square,0.493428,0.700747,250.395774,0.044159
